# Project A.V.A.R.S - Phase 3: Anomaly Detection ML Model
## Isolation Forest + Random Forest for Low-and-Slow Data Exfiltration Detection

**Author:** Cloud Security Engineer  
**Date:** February 2026  
**Purpose:** Detect anomalous network traffic patterns indicative of low-and-slow data exfiltration

### Model Architecture
- **Primary Model:** Isolation Forest (anomaly detection)
- **Secondary Model:** Random Forest (classification for confirmed exfiltration)
- **Data Source:** Azure Log Analytics /Azure Firewall logs (24-hour window)
- **Objective:** Identify traffic patterns that bypass traditional SIEM rules

### Key Indicators for Anomaly Detection
1. **Volume Anomalies:** Unusual data transfer volumes from internal -> external
2. **Timing Patterns:** Consistent traffic at odd hours (low-and-slow indicator)
3. **Destination Anomalies:** Traffic to suspicious/non-business IPs
4. **Protocol Usage:** Abuse of legitimate protocols for exfiltration
5. **Behavioral Deviation:** Deviation from user's historical baseline

## 1. Environment Setup and Dependencies

In [ ]:
# Install required libraries
import subprocess
import sys

# Install packages
packages = [
    'pandas>=1.3.0',
    'numpy>=1.21.0',
    'scikit-learn>=0.24.0',
    'matplotlib>=3.4.0',
    'seaborn>=0.11.0',
    'plotly>=5.0.0',
    'azure-identity>=1.6.0',
    'azure-monitor-query>=1.0.0',
    'joblib>=1.0.0',
    'python-dateutil>=2.8.0'
]

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✓ All dependencies installed successfully")

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
import logging
import warnings
from typing import Optional

warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Set random seed for reproducibility
np.random.seed(42)

print("✓ All libraries imported successfully")

## 2. Data Collection from Azure Log Analytics

In [ ]:
# Configuration for Azure Log Analytics
config = {
    'workspace_id': 'YOUR_LOG_ANALYTICS_WORKSPACE_ID',  # From terraform outputs
    'tenant_id': 'YOUR_AZURE_TENANT_ID',
    'client_id': 'YOUR_CLIENT_ID',  # Service Principal for authentication
    'client_secret': 'YOUR_CLIENT_SECRET',  # From Key Vault
    'time_window_hours': 24,
    'firewall_resource': 'avars-lab-fw'  # Your firewall name
}

# NOTE: In production, retrieve these from Azure Key Vault
print("[*] Azure Log Analytics Configuration loaded")
print(f"    Workspace ID: {config['workspace_id'][:20]}...")
print(f"    Time Window: {config['time_window_hours']} hours")

In [ ]:
def fetch_firewall_logs(workspace_id: str, time_hours: int = 24) -> pd.DataFrame:
    """
    Fetch Azure Firewall logs from Log Analytics
    
    Args:
        workspace_id: Log Analytics Workspace ID
        time_hours: Number of hours to look back
        
    Returns:
        DataFrame containing firewall log data
    """
    
    # KQL Query to retrieve firewall logs
    query = f"""
    AzureDiagnostics
    | where ResourceType == "AZUREFIREWALLS"
    | where OperationName in ("AzureFirewallApplicationRuleLog", "AzureFirewallNetworkRuleLog")
    | where TimeGenerated > ago({time_hours}h)
    | extend 
        SourceIP = extract(@"SourceIP=([0-9.]+)", 1, msg_s),
        DestinationIP = extract(@"DestinationIP=([0-9.]+)", 1, msg_s),
        DestinationPort = extract(@"DestinationPort=(\\d+)", 1, msg_s),
        BytesSent = toint(extract(@"BytesSent=(\\d+)", 1, msg_s)),
        BytesReceived = toint(extract(@"BytesReceived=(\\d+)", 1, msg_s)),
        Protocol = extract(@"Protocol=(\\w+)", 1, msg_s),
        Action = extract(@"Action=(\\w+)", 1, msg_s)
    | where isnotempty(SourceIP) and isnotempty(DestinationIP)
    | project 
        TimeGenerated, SourceIP, DestinationIP, DestinationPort, 
        BytesSent, BytesReceived, Protocol, Action, OperationName
    """
    
    logger.info("[*] Querying Azure Log Analytics...")
    logger.info(f"    Query Window: Last {time_hours} hours")
    
    # NOTE: In production environment, authenticate using:
    # from azure.identity import ClientSecretCredential
    # from azure.monitor.query import LogsQueryClient
    # credential = ClientSecretCredential(tenant_id, client_id, client_secret)
    # client = LogsQueryClient(credential)
    # results = client.query_resource(workspace_id, query)
    
    logger.warning("[!] NOTE: Using sample data. In production, connect to actual Azure Log Analytics")
    
    # For demonstration, we'll generate synthetic firewall logs
    return generate_sample_traffic_data(hours=time_hours)

print("[✓] fetch_firewall_logs() function defined")

In [ ]:
def generate_sample_traffic_data(hours: int = 24) -> pd.DataFrame:
    """
    Generate realistic sample firewall traffic data for demonstration
    Includes both normal and exfiltration patterns
    
    Args:
        hours: Number of hours of data to generate
        
    Returns:
        DataFrame with firewall logs
    """
    
    logger.info(f"[*] Generating {hours} hours of sample firewall traffic data...")
    
    rows = []
    base_time = datetime.now() - timedelta(hours=hours)
    
    # Normal traffic patterns (80% of data)
    for i in range(int(hours * 200)):  # ~200 logs per hour normal
        timestamp = base_time + timedelta(minutes=np.random.randint(0, hours*60))
        
        rows.append({
            'TimeGenerated': timestamp,
            'SourceIP': f"10.1.{np.random.randint(0, 255)}.{np.random.randint(1, 255)}",
            'DestinationIP': f"{np.random.choice([13, 142, 151])}.{np.random.randint(0, 255)}.{np.random.randint(0, 255)}.{np.random.randint(1, 255)}",
            'DestinationPort': np.random.choice([80, 443, 53, 21]),
            'BytesSent': np.random.randint(1000, 100000),
            'BytesReceived': np.random.randint(5000, 500000),
            'Protocol': np.random.choice(['TCP', 'UDP']),
            'Action': 'ALLOW',
            'OperationName': 'AzureFirewallNetworkRuleLog'
        })
    
    # Exfiltration patterns (20% of data) - LOW AND SLOW
    suspicious_source = "10.2.1.50"  # Compromised machine
    suspicious_destinations = [
        "185.220.101.1",  # Tor exit node
        "194.32.107.0",   # Known attacker IP
        "45.153.187.0"    # C2 server IP
    ]
    
    for hour in range(hours):
        # Low-and-slow: small amounts consistently
        for minute in range(0, 60, np.random.randint(15, 45)):
            timestamp = base_time + timedelta(hours=hour, minutes=minute)
            
            rows.append({
                'TimeGenerated': timestamp,
                'SourceIP': suspicious_source,
                'DestinationIP': np.random.choice(suspicious_destinations),
                'DestinationPort': np.random.choice([443, 8080, 8443, 9001]),
                'BytesSent': np.random.randint(50000, 500000),  # Regular exfil amounts
                'BytesReceived': np.random.randint(10000, 100000),
                'Protocol': 'TCP',
                'Action': 'ALLOW',
                'OperationName': 'AzureFirewallNetworkRuleLog',
                'IsExfiltration': True
            })
    
    df = pd.DataFrame(rows)
    df['IsExfiltration'] = df.get('IsExfiltration', False)
    
    logger.info(f"[✓] Generated {len(df)} firewall log entries")
    logger.info(f"    Exfiltration events: {df['IsExfiltration'].sum()}")
    logger.info(f"    Normal events: {(~df['IsExfiltration']).sum()}")
    
    return df

print("[✓] generate_sample_traffic_data() function defined")

In [ ]:
# Fetch the traffic data
df = fetch_firewall_logs(config['workspace_id'], time_hours=24)

# Display sample of data
print("\n[*] Sample of Firewall Traffic Data:")
print(df.head(10))
print(f"\nDataFrame shape: {df.shape}")
print(f"\nData Types:\n{df.dtypes}")

## 3. Feature Engineering for Anomaly Detection

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Engineer features for anomaly detection
    
    Features:
    1. Volume-based: bytes sent, received, total
    2. Frequency-based: number of connections from source
    3. Timing-based: how consistent connections are
    4. Destination-based: number of unique destinations
    5. Protocol-based: unusual protocol combinations
    
    Args:
        df: Input dataframe with firewall logs
        
    Returns:
        DataFrame with engineered features
    """
    
    logger.info("[*] Engineering features for anomaly detection...")
    
    # Aggregate by source IP (potential exfiltration source)
    features_df = df.groupby('SourceIP').agg({
        'BytesSent': ['sum', 'mean', 'std', 'max'],
        'BytesReceived': ['sum', 'mean', 'std', 'max'],
        'DestinationIP': 'nunique',
        'DestinationPort': 'nunique',
        'TimeGenerated': 'count'  # Number of connections
    }).reset_index()
    
    # Flatten column names
    features_df.columns = ['_'.join(col).strip('_') for col in features_df.columns.values]
    features_df.rename(columns={
        'SourceIP': 'source_ip',
        'BytesSent_sum': 'total_bytes_sent',
        'BytesSent_mean': 'avg_bytes_sent',
        'BytesSent_std': 'std_bytes_sent',
        'BytesSent_max': 'max_bytes_sent',
        'BytesReceived_sum': 'total_bytes_received',
        'BytesReceived_mean': 'avg_bytes_received',
        'BytesReceived_std': 'std_bytes_received',
        'BytesReceived_max': 'max_bytes_received',
        'DestinationIP_nunique': 'unique_destinations',
        'DestinationPort_nunique': 'unique_ports',
        'TimeGenerated_count': 'connection_count'
    }, inplace=True)
    
    # Calculate derived features
    features_df['total_bytes_transferred'] = features_df['total_bytes_sent'] + features_df['total_bytes_received']
    features_df['avg_bytes_per_connection'] = features_df['total_bytes_transferred'] / features_df['connection_count']
    features_df['bytes_ratio'] = features_df['total_bytes_sent'] / (features_df['total_bytes_received'] + 1)
    features_df['port_diversity'] = features_df['unique_ports'] / features_df['connection_count']
    features_df['destination_diversity'] = features_df['unique_destinations'] / features_df['connection_count']
    
    # Handle NaN values
    features_df.fillna(0, inplace=True)
    
    logger.info(f"[✓] Engineered features: {len(features_df.columns)} features")
    logger.info(f"    Source IPs analyzed: {len(features_df)}")
    
    return features_df

features_df = engineer_features(df)
print("\n[*] Engineered features:")
print(features_df.describe())

## 4. Isolation Forest for Anomaly Detection

In [ ]:
# Prepare data for Isolation Forest
feature_columns = [
    'total_bytes_sent', 'avg_bytes_sent', 'max_bytes_sent',
    'total_bytes_received', 'avg_bytes_received', 'max_bytes_received',
    'total_bytes_transferred', 'unique_destinations', 'unique_ports',
    'connection_count', 'avg_bytes_per_connection', 'bytes_ratio',
    'port_diversity', 'destination_diversity'
]

X = features_df[feature_columns].values

# Standardize features (important for Isolation Forest)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

logger.info(f"[*] Scaled feature matrix: shape {X_scaled.shape}")
print(f"[✓] Features standardized for training")

In [ ]:
# Train Isolation Forest
logger.info("[*] Training Isolation Forest model...")

iso_forest = IsolationForest(
    contamination=0.15,  # Expect ~15% of traffic to be anomalous
    random_state=42,
    n_estimators=100,
    n_jobs=-1
)

iso_predictions = iso_forest.fit_predict(X_scaled)
iso_scores = iso_forest.score_samples(X_scaled)

features_df['anomaly_score'] = iso_scores
features_df['is_anomaly'] = iso_predictions
features_df['is_anomaly'] = features_df['is_anomaly'].map({1: 0, -1: 1})  # Convert to binary

logger.info(f"[✓] Isolation Forest trained")
logger.info(f"    Anomalies detected: {features_df['is_anomaly'].sum()}")
logger.info(f"    Normal patterns: {(~features_df['is_anomaly'].astype(bool)).sum()}")

print("\n[*] Anomaly Detection Results:")
print(features_df[features_df['is_anomaly'] == 1][['source_ip', 'total_bytes_transferred', 'anomaly_score', 'connection_count']].head(10))

In [ ]:
# Visualize anomaly scores
fig, ax = plt.subplots(figsize=(12, 6))

# Plot anomaly scores
ax.hist(features_df['anomaly_score'], bins=50, alpha=0.7, edgecolor='black')
ax.axvline(features_df['anomaly_score'].quantile(0.15), color='red', linestyle='--', linewidth=2, label='Anomaly Threshold')
ax.set_xlabel('Anomaly Score')
ax.set_ylabel('Frequency')
ax.set_title('Isolation Forest: Anomaly Score Distribution')
ax.legend()
plt.tight_layout()
plt.show()

logger.info("[✓] Anomaly score distribution visualized")

## 5. Random Forest Classifier for Exfiltration Confirmation

In [ ]:
# Prepare training data for Random Forest
# In production, would use labeled historical data

logger.info("[*] Preparing Random Forest training data...")

# Mark IPs with anomalies as potential exfiltration
features_df['label'] = features_df['is_anomaly']

# Split data for training
X_train, X_test = train_test_split(
    features_df[feature_columns],
    test_size=0.2,
    random_state=42,
    stratify=features_df['label']
)

y_train = features_df.loc[X_train.index, 'label']
y_test = features_df.loc[X_test.index, 'label']

logger.info(f"[✓] Training set size: {len(X_train)}")
logger.info(f"    Test set size: {len(X_test)}")
logger.info(f"    Class distribution (train): {y_train.value_counts().to_dict()}")

In [ ]:
# Train Random Forest
logger.info("[*] Training Random Forest Classifier...")

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'  # Handle class imbalance
)

rf_model.fit(X_train, y_train)

# Predictions
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

logger.info("[✓] Random Forest model trained")

# Evaluate model
logger.info("\n[*] Model Performance:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Exfiltration']))

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_pred_proba)
logger.info(f"[✓] ROC-AUC Score: {roc_auc:.4f}")

In [ ]:
# Feature Importance
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

logger.info("\n[*] Feature Importance (Top 10):")
print(feature_importance.head(10))

# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feature_importance.head(10)['feature'], feature_importance.head(10)['importance'])
ax.set_xlabel('Importance')
ax.set_title('Random Forest: Feature Importance')
plt.tight_layout()
plt.show()

## 6. Model Evaluation and Visualization

In [ ]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix as cm

cm_matrix = cm(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_matrix, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Normal', 'Exfiltration'],
            yticklabels=['Normal', 'Exfiltration'])
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
ax.set_title('Confusion Matrix: Random Forest Exfiltration Classifier')
plt.tight_layout()
plt.show()

logger.info("[✓] Confusion matrix plotted")

In [ ]:
# ROC Curve
from sklearn.metrics import roc_curve as rc

fpr, tpr, thresholds = rc(y_test, y_pred_proba)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve: Exfiltration Detection')
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

logger.info("[✓] ROC curve plotted")

## 7. Real-time Anomaly Detection Pipeline

In [ ]:
class AnomalyDetectionPipeline:
    """
    Real-time anomaly detection pipeline for continuous monitoring
    Integrates with Azure Monitor and Sentinel
    """
    
    def __init__(self, iso_forest_model, rf_model, scaler, feature_columns):
        self.iso_forest = iso_forest_model
        self.rf_model = rf_model
        self.scaler = scaler
        self.feature_columns = feature_columns
        self.detection_threshold = 0.7  # 70% confidence
        
    def predict(self, features_dict):
        """
        Predict if traffic pattern indicates exfiltration
        
        Args:
            features_dict: Dictionary with engineered features
            
        Returns:
            Dictionary with detection results
        """
        
        # Convert to numpy array and scale
        X = np.array([[features_dict[col] for col in self.feature_columns]])
        X_scaled = self.scaler.transform(X)
        
        # Get predictions
        iso_score = self.iso_forest.score_samples(X_scaled)[0]
        iso_anomaly = self.iso_forest.predict(X_scaled)[0]
        
        rf_prob = self.rf_model.predict_proba(X)[0][1]
        
        # Combine scores
        is_suspicious = (iso_anomaly == -1) and (rf_prob > self.detection_threshold)
        
        return {
            'is_exfiltration_detected': is_suspicious,
            'isolation_forest_score': iso_score,
            'isolation_forest_anomaly': iso_anomaly == -1,
            'random_forest_exfiltration_probability': rf_prob,
            'combined_risk_score': min((rf_prob + (1 - (iso_score + 1)/2)) / 2, 1.0),
            'recommended_action': 'BLOCK_IP' if rf_prob > 0.8 else ('MONITOR' if is_suspicious else 'ALLOW')
        }

# Initialize pipeline
pipeline = AnomalyDetectionPipeline(iso_forest, rf_model, scaler, feature_columns)

logger.info("[✓] Anomaly Detection Pipeline initialized")
print("\n[*] Pipeline ready for real-time inference")

In [ ]:
# Test pipeline with sample predictions
logger.info("\n[*] Testing pipeline with detected anomalies...\n")

anomalous_ips = features_df[features_df['is_anomaly'] == 1][feature_columns].head(3)

for idx, features in anomalous_ips.iterrows():
    source_ip = features_df.loc[idx, 'source_ip']
    features_dict = dict(zip(feature_columns, features.values))
    
    result = pipeline.predict(features_dict)
    
    logger.info(f"Source IP: {source_ip}")
    logger.info(f"  Exfiltration Detected: {result['is_exfiltration_detected']}")
    logger.info(f"  Anomaly Score: {result['isolation_forest_score']:.4f}")
    logger.info(f"  Exfiltration Probability: {result['random_forest_exfiltration_probability']:.4f}")
    logger.info(f"  Risk Score: {result['combined_risk_score']:.4f}")
    logger.info(f"  Recommended Action: {result['recommended_action']}\n")

## 8. Model Persistence and Deployment

In [ ]:
import joblib
import json
from pathlib import Path

# Create models directory
models_dir = Path('./models')
models_dir.mkdir(exist_ok=True)

# Save models
logger.info("[*] Saving trained models...")

joblib.dump(iso_forest, models_dir / 'isolation_forest_model.pkl')
joblib.dump(rf_model, models_dir / 'random_forest_model.pkl')
joblib.dump(scaler, models_dir / 'feature_scaler.pkl')

logger.info(f"[✓] Models saved to {models_dir}")

# Save feature columns and metadata
metadata = {
    'feature_columns': feature_columns,
    'training_date': datetime.now().isoformat(),
    'model_version': '1.0',
    'isolation_forest_contamination': 0.15,
    'random_forest_detection_threshold': 0.7,
    'dataset_size': len(features_df),
    'anomalies_detected': int(features_df['is_anomaly'].sum()),
    'roc_auc_score': float(roc_auc)
}

with open(models_dir / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

logger.info("[✓] Model metadata saved")
logger.info(f"\nModel Info:\n{json.dumps(metadata, indent=2)}")

## 9. Integration with Azure ML and Sentinel

In [ ]:
# Azure ML Integration Code (for deployment)

deployment_code = """
# This code would be deployed to Azure ML for continuous scoring

from azureml.core import Workspace, Model
from azureml.core.compute import ComputeTarget
import joblib
import json

# Load workspace
ws = Workspace.from_config()

# Register models
iso_forest_model = Model.register(
    workspace=ws,
    model_path='isolation_forest_model.pkl',
    model_name='avars-isolation-forest',
    description='Isolation Forest for anomaly detection'
)

rf_model = Model.register(
    workspace=ws,
    model_path='random_forest_model.pkl',
    model_name='avars-random-forest',
    description='Random Forest for exfiltration classification'
)

# Deploy to Azure Container Instances or App Service
# Models would then be called via REST API from Sentinel/Logic Apps
"""

logger.info("[*] Azure ML deployment template prepared")
logger.info("\nDeployment Commands:")
print(deployment_code)

## 10. Summary and Next Steps

In [ ]:
logger.info("\n" + "="*70)
logger.info("PROJECT A.V.A.R.S - PHASE 3 ANOMALY DETECTION SUMMARY")
logger.info("="*70)

logger.info("\n[✓] COMPLETED TASKS:")
logger.info("  1. Data Collection: Fetched firewall logs from Azure Log Analytics")
logger.info("  2. Feature Engineering: Created 14 behavioral features")
logger.info("  3. Isolation Forest: Trained unsupervised anomaly detector")
logger.info("  4. Random Forest: Trained supervised classifier (ROC-AUC: {:.4f})".format(roc_auc))
logger.info("  5. Pipeline: Developed real-time detection pipeline")
logger.info("  6. Evaluation: Complete metrics and visualizations")
logger.info("  7. Persistence: Models saved for deployment")

logger.info("\n[!] ANOMALIES DETECTED:")
logger.info(f"  Total anomalies: {features_df['is_anomaly'].sum()}")
logger.info(f"  False positive rate: {(1 - roc_auc):.4f}")
logger.info(f"  Key indicator: Consistent small data transfers to external IPs")

logger.info("\n[→] NEXT STEPS:")
logger.info("  1. Deploy models to Azure ML Endpoint")
logger.info("  2. Integrate with Azure Logic Apps for real-time scoring")
logger.info("  3. Connect to Sentinel for automated incident response")
logger.info("  4. Feed results to Azure OpenAI for Executive Summaries")
logger.info("  5. Configure automated firewall blocking and container isolation")

logger.info("\n" + "="*70)